# Session 1 Notebook - Introduction

## 1) Overview main() function


GETTSIM has one main entry point. This is the `main()` function. We unpack its inputs in this minimal tutorial step by step and use it to create a workflow for simulation.

In [1]:
# Command for installing gettsim (only use on Colab)

#! pip install gettsim
# !pip install git+https://github.com/ttsim-dev/gettsim-personas.git

In [2]:
from gettsim import main

# main(
#     main_target=
#     policy_date_str=
#     input_data=
#     tt_targets=
# )

### Core inputs and additional inputs

#### `policy_date_str` (put this first)

- Defines the legal policy environment for the run.
- Use ISO format: `YYYY-MM-DD` (for example `2025-01-01`).
- In most class examples, keep this fixed while changing inputs/targets.

#### `main_target` options (what `main()` returns)

- `MainTarget.results.tree`: nested tree output (good for conceptual structure).
- `MainTarget.results.df_with_mapper`: tabular output (good for plotting and checks).
- `MainTarget.results.df_with_nested_columns`: DataFrame with nested-style columns.
- `MainTarget.processed_data`: mapped input data in GETTSIM qname form (very useful for debugging mapper issues).
- `MainTarget.input_data`: the input object used by `main()`.
- `MainTarget.tt_targets`: the resolved targets object.

#### `input_data` options

Provide one of the following forms:
- `InputData.df_and_mapper(df=..., mapper=...)`
- `InputData.tree(tree=...)`
- `InputData.df_with_nested_columns(df=...)`

#### `tt_targets` options

- Standard usage: `TTTargets(tree=...)` where tree leaves are `None`.
- Each leaf path corresponds to a requested qname target (for example `einkommensteuer__betrag_y_sn`).
- Keep target sets small at first to reduce required roots and improve error readability.

#### Additional inputs (used in specific situations)

- `backend`: choose numerical backend (`numpy` or `jax`).
- `rounding`: switch legal rounding behavior on/off.
- `include_fail_nodes` / `include_warn_nodes`: control strict diagnostics and warnings.
- `evaluation_date_str`: optional separate evaluation date (if different from policy date).

## 2) Use main() to explore gettsim 

#### Step 1: Define targets

In [3]:
from gettsim import TTTargets

# Small target set on purpose: this keeps diagnostics interpretable.
# Each `None` is a leaf marker ("request this target"), not missing data.

targets_small = {
    "einkommensteuer": {"betrag_y_sn": None},
    "solidaritätszuschlag": {"betrag_y_sn": None},
}

tt_targets_small = TTTargets(tree=targets_small)

### Step 2: Input data

In [4]:
import pandas as pd

# Minimal toy data for learning diagnostics.
# This table is intentionally incomplete for full simulation runs.
# `capital_income_yearly` is a custom name and will be mapped later.

df_example = pd.DataFrame(
    {
        "p_id": [0, 1, 2],
        "hh_id": [0, 0, 0],
        "alter": [35, 34, 8],
        "arbeitsstunden_w": [40.0, 30.0, 0.0],
        "bruttolohn_m": [4200.0, 2400.0, 0.0],
        "capital_income_yearly": [120.0, 0.0, 0.0],
    }
)

df_example

,p_id,hh_id,alter,arbeitsstunden_w,bruttolohn_m,capital_income_yearly
0,0,0,35,40.0,4200.0,120.0
1,1,0,34,30.0,2400.0,0.0
2,2,0,8,0.0,0.0,0.0


#### Step 3: Mapper (how columns map into GETTSIM names/tree)

In [5]:
from gettsim import InputData

mapper_example = {
    "p_id": "p_id",
    "hh_id": "hh_id",
    "alter": "alter",
    "arbeitsstunden_w": "arbeitsstunden_w",
    "einnahmen": {
        "bruttolohn_m": "bruttolohn_m",
        "kapitalerträge_y": "capital_income_yearly",
    },
}

input_data_df_mapper = InputData.df_and_mapper(df=df_example, mapper=mapper_example)


#### Step 4: Run and inspect missing required inputs

In [6]:
from gettsim import MainTarget

# Intentional diagnostic run.
# We expect this to fail and use the error message as a requirements list.
# This is the core workflow for building inputs target-by-target.

try:
    main(
        main_target=MainTarget.results.tree,
        policy_date_str="2025-01-01",
        input_data=input_data_df_mapper,
        tt_targets=tt_targets_small,
    )
except Exception as err:
    print(err)

The following data columns are missing.

[
    "('sozialversicherung', 'rente', 'bezieht_rente')",
    "('familie', 'p_id_ehepartner')",
    "('familie', 'p_id_elternteil_1')",
    "('familie', 'p_id_elternteil_2')",
    "('einkommensteuer', 'gemeinsam_veranlagt')",
    "('behinderungsgrad',)",
    "('familie', 'alleinerziehend')",
    "('einkommensteuer', 'einkünfte', 'aus_forst_und_landwirtschaft', 'betrag_y')",
    "('einkommensteuer', 'einkünfte', 'aus_gewerbebetrieb', 'betrag_y')",
    "('einkommensteuer', 'einkünfte', 'aus_selbstständiger_arbeit', 'betrag_y')",
    "('einkommensteuer', 'einkünfte', 'aus_vermietung_und_verpachtung', 'betrag_y')",
    "('einkommensteuer', 'einkünfte', 'sonstige', 'alle_weiteren_y')",
    "('sozialversicherung', 'rente', 'jahr_renteneintritt')",
    "('einnahmen', 'renten', 'geförderte_private_vorsorge_m')",
    "('einnahmen', 'renten', 'sonstige_private_vorsorge_m')",
    "('einnahmen', 'renten', 'betriebliche_altersvorsorge_m')",
    "('einnahmen'

#### Step 5: Add a few columns that the error asks for

In [7]:
df_example_more = df_example.assign(
    geburtsjahr=[1990, 1991, 2017],
    behinderungsgrad=[0, 0, 0],
    familie_p_id_ehepartner=[1, 0, -1],
    familie_p_id_elternteil_1=[-1, -1, 0],
    familie_p_id_elternteil_2=[-1, -1, 1],
    einkommensteuer_gemeinsam_veranlagt=[True, True, False],
)

df_example_more

,p_id,hh_id,alter,arbeitsstunden_w,bruttolohn_m,capital_income_yearly,geburtsjahr,behinderungsgrad,familie_p_id_ehepartner,familie_p_id_elternteil_1,familie_p_id_elternteil_2,einkommensteuer_gemeinsam_veranlagt
0,0,0,35,40.0,4200.0,120.0,1990,0,1,-1,-1,True
1,1,0,34,30.0,2400.0,0.0,1991,0,0,-1,-1,True
2,2,0,8,0.0,0.0,0.0,2017,0,-1,0,1,False


In [8]:
# Extended mapper.

mapper_more = {
    "p_id": "p_id",
    "hh_id": "hh_id",
    "alter": "alter",
    "geburtsjahr": "geburtsjahr",
    "behinderungsgrad": "behinderungsgrad",
    "arbeitsstunden_w": "arbeitsstunden_w",
    "einnahmen": {
        "bruttolohn_m": "bruttolohn_m",
        "kapitalerträge_y": "capital_income_yearly",
    },
    "familie": {
        "p_id_ehepartner": "familie_p_id_ehepartner",
        "p_id_elternteil_1": "familie_p_id_elternteil_1",
        "p_id_elternteil_2": "familie_p_id_elternteil_2",
        "alleinerziehend": False,
    },
    "einkommensteuer": {
        "gemeinsam_veranlagt": "einkommensteuer_gemeinsam_veranlagt",
    },
}

input_data_more = InputData.df_and_mapper(df=df_example_more, mapper=mapper_more)


try:
    main(
        main_target=MainTarget.results.tree,
        policy_date_str="2025-01-01",
        input_data=input_data_more,
        tt_targets=TTTargets(tree=targets_small),
    )
except Exception as err:
    print(err)

The following data columns are missing.

[
    "('sozialversicherung', 'rente', 'bezieht_rente')",
    "('einkommensteuer', 'einkünfte', 'aus_forst_und_landwirtschaft', 'betrag_y')",
    "('einkommensteuer', 'einkünfte', 'aus_gewerbebetrieb', 'betrag_y')",
    "('einkommensteuer', 'einkünfte', 'aus_selbstständiger_arbeit', 'betrag_y')",
    "('einkommensteuer', 'einkünfte', 'aus_vermietung_und_verpachtung', 'betrag_y')",
    "('einkommensteuer', 'einkünfte', 'sonstige', 'alle_weiteren_y')",
    "('sozialversicherung', 'rente', 'jahr_renteneintritt')",
    "('einnahmen', 'renten', 'geförderte_private_vorsorge_m')",
    "('einnahmen', 'renten', 'sonstige_private_vorsorge_m')",
    "('einnahmen', 'renten', 'betriebliche_altersvorsorge_m')",
    "('einnahmen', 'renten', 'aus_berufsständischen_versicherungen_m')",
    "('einkommensteuer', 'einkünfte', 'sonstige', 'rente', 'alter_beginn_leistungsbezug_sonstige_private_vorsorge')",
    "('einkommensteuer', 'abzüge', 'kinderbetreuungskosten_m'

#### Step 6: Overwrite a whole branch (`einnahmen -> renten`)

If we are not analyzing pensions here, we can provide a full rent-related block with neutral values (mostly zeros).

The goal is not to get all inputs perfect immediately. The goal is to show
how supplying one coherent branch can cut off many required roots in the DAG.

In [9]:
# Here we overwrite the pension branch through data columns.
#
# Why data columns (instead of mapper constants)?
# - it is visible in the dataframe
# - students see exactly which variables are provided
# - easier to adapt values later for exercises
#
# We include both direct `einnahmen -> renten` leaves and additional
# rente-related anchors that are used in persona-style setups.

df_example_rente = df_example_more.assign(
    renten_gefoerderte_private_vorsorge_m=[0.0, 0.0, 0.0],
    renten_sonstige_private_vorsorge_m=[0.0, 0.0, 0.0],
    renten_betriebliche_altersvorsorge_m=[0.0, 0.0, 0.0],
    renten_aus_berufsstaendischen_versicherungen_m=[0.0, 0.0, 0.0],
    einkommensteuer_sonstige_rente_betrag_m=[0.0, 0.0, 0.0],
    sozialversicherung_kranken_bemessungsgrundlage_rente_m=[0.0, 0.0, 0.0],
    einkommensteuer_beitrag_private_rentenversicherung_m=[0.0, 0.0, 0.0],
    sozialversicherung_rente_bezieht_rente=[False, False, False],
)

mapper_with_rente_override = {
    "p_id": "p_id",
    "hh_id": "hh_id",
    "alter": "alter",
    "geburtsjahr": "geburtsjahr",
    "behinderungsgrad": "behinderungsgrad",
    "arbeitsstunden_w": "arbeitsstunden_w",
    "einnahmen": {
        "bruttolohn_m": "bruttolohn_m",
        "kapitalerträge_y": "capital_income_yearly",
        "renten": {
            "geförderte_private_vorsorge_m": "renten_gefoerderte_private_vorsorge_m",
            "sonstige_private_vorsorge_m": "renten_sonstige_private_vorsorge_m",
            "betriebliche_altersvorsorge_m": "renten_betriebliche_altersvorsorge_m",
            "aus_berufsständischen_versicherungen_m": "renten_aus_berufsstaendischen_versicherungen_m",
        },
    },
    "familie": {
        "p_id_ehepartner": "familie_p_id_ehepartner",
        "p_id_elternteil_1": "familie_p_id_elternteil_1",
        "p_id_elternteil_2": "familie_p_id_elternteil_2",
        "alleinerziehend": False,
    },
    "einkommensteuer": {
        "gemeinsam_veranlagt": "einkommensteuer_gemeinsam_veranlagt",
        "abzüge": {
            "beitrag_private_rentenversicherung_m": "einkommensteuer_beitrag_private_rentenversicherung_m",
        },
        "einkünfte": {
            "sonstige": {
                "rente": {
                    "betrag_m": "einkommensteuer_sonstige_rente_betrag_m",
                },
            },
        },
    },
    "sozialversicherung": {
        "kranken": {
            "beitrag": {
                "bemessungsgrundlage_rente_m": "sozialversicherung_kranken_bemessungsgrundlage_rente_m",
            },
        },
        "rente": {
            "bezieht_rente": "sozialversicherung_rente_bezieht_rente",
        },
    },
}

input_data_with_rente_override = InputData.df_and_mapper(
    df=df_example_rente,
    mapper=mapper_with_rente_override,
)

try:
    main(
        main_target=MainTarget.results.tree,
        policy_date_str="2025-01-01",
        input_data=input_data_with_rente_override,
        tt_targets=tt_targets_small,
        include_warn_nodes=False,
    )
except Exception as err:
    print(str(err))

The following data columns are missing.

[
    "('einkommensteuer', 'einkünfte', 'aus_forst_und_landwirtschaft', 'betrag_y')",
    "('einkommensteuer', 'einkünfte', 'aus_gewerbebetrieb', 'betrag_y')",
    "('einkommensteuer', 'einkünfte', 'aus_selbstständiger_arbeit', 'betrag_y')",
    "('einkommensteuer', 'einkünfte', 'aus_vermietung_und_verpachtung', 'betrag_y')",
    "('einkommensteuer', 'einkünfte', 'sonstige', 'alle_weiteren_y')",
    "('einkommensteuer', 'abzüge', 'kinderbetreuungskosten_m')",
    "('einkommensteuer', 'abzüge', 'p_id_kinderbetreuungskostenträger')",
    "('einkommensteuer', 'einkünfte', 'ist_hauptberuflich_selbstständig')",
    "('sozialversicherung', 'kranken', 'beitrag', 'privat_versichert')",
    "('sozialversicherung', 'pflege', 'beitrag', 'hat_kinder')",
    "('kindergeld', 'p_id_empfänger')",
    "('kindergeld', 'in_ausbildung')",
]



## 3) Persona as a complete input package

Personas provide a coherent synthetic household and target setup.

In this part we do four things: inspect persona input data as a dataframe, run with a small target set, inspect the persona target list, and then run with persona targets.

In [10]:
from gettsim_personas import einkommensteuer_sozialabgaben
import dags.tree as dt

# Instantiate one stylized household with fixed policy date.
# The persona object bundles both an input tree and a target tree.

persona = einkommensteuer_sozialabgaben.Couple1Child(policy_date_str="2025-01-01")
print(persona.description)

Persona to compute income taxes and social insurance contributions. Jointly
        taxed married couple with one child. All transfers are set to zero; don't use
        this persona for low- to mid-income households, as they may be eligible for
        (means-tested) transfers.


In [11]:
# Flatten the persona input tree so we can inspect it in dataframe form.
# This is useful to compare persona inputs with our custom dataframe workflow.

persona_input_df = pd.DataFrame(dt.flatten_to_qnames(persona.input_data_tree))
persona_input_df

,alter,arbeitsstunden_w,behinderungsgrad,einkommensteuer__abzüge__beitrag_private_rentenversicherung_m,einkommensteuer__abzüge__kinderbetreuungskosten_m,einkommensteuer__abzüge__p_id_kinderbetreuungskostenträger,einkommensteuer__einkünfte__aus_forst_und_landwirtschaft__betrag_m,einkommensteuer__einkünfte__aus_gewerbebetrieb__betrag_m,einkommensteuer__einkünfte__aus_nichtselbstständiger_arbeit__tatsächliche_werbungskosten_y,einkommensteuer__einkünfte__aus_selbstständiger_arbeit__betrag_m,...,familie__p_id_elternteil_1,familie__p_id_elternteil_2,geburtsjahr,hh_id,kindergeld__in_ausbildung,kindergeld__p_id_empfänger,p_id,sozialversicherung__kranken__beitrag__bemessungsgrundlage_rente_m,sozialversicherung__kranken__beitrag__privat_versichert,sozialversicherung__pflege__beitrag__hat_kinder
0,30,39,0,0,0.0,-1,0,0,0,0,...,-1,-1,1995,0,False,-1,0,0,False,True
1,30,39,0,0,0.0,-1,0,0,0,0,...,-1,-1,1995,0,False,-1,1,0,False,True
2,10,0,0,0,100.0,0,0,0,0,0,...,0,1,2015,0,False,0,2,0,False,False


In [12]:
result_small_targets_df = main(
    main_target=MainTarget.results.df_with_nested_columns,
    policy_date_str="2025-01-01",
    input_data=InputData.tree(tree=persona.input_data_tree),
    tt_targets=tt_targets_small,
    include_warn_nodes=False,
)

result_small_targets_df

,einkommensteuer,solidaritätszuschlag
,betrag_y_sn,betrag_y_sn
p_id,,
0,7120.0,0.0
1,7120.0,0.0
2,0.0,0.0


### Persona targets

In [13]:
persona.tt_targets_tree

{'einkommensteuer': {'betrag_m_sn': None},
 'kindergeld': {'betrag_m_hh': None},
 'solidaritätszuschlag': {'betrag_m_sn': None},
 'sozialversicherung': {'beiträge_versicherter_m_hh': None}}

In [14]:
result_persona_targets_df = main(
    main_target=MainTarget.results.df_with_nested_columns,
    policy_date_str="2025-01-01",
    input_data=InputData.tree(tree=persona.input_data_tree),
    tt_targets=TTTargets(tree=persona.tt_targets_tree),
    include_warn_nodes=False,
)

result_persona_targets_df

,einkommensteuer,kindergeld,solidaritätszuschlag,sozialversicherung
,betrag_m_sn,betrag_m_hh,betrag_m_sn,beiträge_versicherter_m_hh
p_id,,,,
0,593.333333,255,0.0,1257.0
1,593.333333,255,0.0,1257.0
2,0.000000,255,0.0,1257.0
